# QAOA checkpoint and resume

This notebook shows:
- the difference between a new `solve()` and continuing with `resume()`;
- a consistent pause after two ADAM iterations;
- inspection without running the solver;
- continuation in a new object and comparison with an uninterrupted run;
- conservative charging after an unfinished attempt;
- the capability matrix for the other implementations.

The main example is CPU with a small statevector. GPU checkpointing is
deliberately not claimed in `0.1.0a10`: the CuPy RNG needs a separate portable
protocol.

In [1]:
from copy import deepcopy
from pathlib import Path
import os
import tempfile

ROOT = Path.cwd()
if not (ROOT / "experiments").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from QA_max_cut_lib import GraphFactory, QAOACPU, RunControl, inspect_checkpoint, restore, tools
from QA_max_cut_lib.core.checkpoint import ResourceJournal
from QA_max_cut_lib.core.checkpoint_capabilities import capabilities

graph = GraphFactory.from_recipe(tools.load_json("configs/graphs/complete_signed_7.json"))
base_config = tools.load_json("configs/solvers/qaoa_cpu_checkpoint.json")
root = Path(tempfile.mkdtemp(prefix="qa_a10_notebook_"))
checkpoint_config = deepcopy(base_config)
checkpoint_config["checkpoint"]["path"] = str(root)
checkpoint_config["output"] = {"report": 0, "logs": 0, "color": "never"}
print("store:", root)


store: C:\Users\Alexey\AppData\Local\Temp\qa_a10_notebook_w_2rnqdc


## A. Uninterrupted run and pause

The uninterrupted and paused runs use the same configuration and seed. Expected
training shots: 320 after two iterations, held readout: 64.

In [2]:
continuous_config = deepcopy(checkpoint_config)
continuous_config.pop("checkpoint", None)
continuous = QAOACPU(continuous_config)
continuous_solution = continuous.solve(graph)
print("continuous:", continuous.status, continuous.resources["shots_used"], continuous.cut_value)

paused = QAOACPU(checkpoint_config)
paused_solution = paused.solve(graph, control=RunControl(pause_after={"optimizer_iterations": 2}))
print("paused:", paused.status, paused.termination_reason)
print("resources:", paused.resources)
print("checkpoint:", paused.last_checkpoint.path)
info = inspect_checkpoint(paused.last_checkpoint.path)
print("phase:", info["phase"], "next:", info["next_operation"])
assert paused.status == "paused"
assert paused.resources["known_spent"] == 320
assert paused.resources["readout_reserved"] == 64


continuous: completed 704 2.0
paused: paused checkpoint_pause
resources: {'shots_used': 320, 'known_spent': 320, 'uncertain_upper_charge': 0, 'total_charged': 320, 'training_shots': 320, 'logical_training_shots': 320, 'readout_shots': 0, 'readout_reserved': 64, 'shots_available': 320, 'objective_evaluations': 10, 'optimizer_steps': 2, 'num_stages': 3, 'num_treads': 1, 'checkpoint_writes': 2}
checkpoint: C:\Users\Alexey\AppData\Local\Temp\qa_a10_notebook_w_2rnqdc\1cab7ad4-275c-4fb0-92b7-30c2afe506b9\gen_000002
phase: training next: {'optimizer_iteration': 3}


## B. Continuation in a new object

`restore()` does not run the solver. Only `resume()` continues the canonical run.

In [3]:
continued = restore(paused.last_checkpoint.path)
final_solution = continued.resume()
print("resumed:", continued.status, continued.resources["shots_used"], continued.cut_value)
print("same run id:", continued.run_id == paused.run_id)
print("same solution:", final_solution == continuous_solution)
print("max angle error:", float(np.max(np.abs(continued.optimal_angles - continuous.optimal_angles))))
assert final_solution == continuous_solution
assert np.allclose(continued.optimal_angles, continuous.optimal_angles)
assert continued.resources["objective_evaluations"] == continuous.resources["objective_evaluations"]
assert continued.resources["shots_used"] == continuous.resources["shots_used"] == 704


resumed: completed 704 2.0
same run id: True
same solution: True
max angle error: 0.0


## C. Unfinished attempt and conservative charging

BEGIN without COMMIT/RESOLVE means the measurement may have happened. The journal
charges the upper bound once and closes the operation. This is not presented as a
proven number of physical shots.

In [4]:
root_b = Path(tempfile.mkdtemp(prefix="qa_a10_b_"))
config_b = deepcopy(checkpoint_config)
config_b["checkpoint"]["path"] = str(root_b)
solver_b = QAOACPU(config_b)
solver_b.solve(graph, control=RunControl(pause_after={"optimizer_iterations": 2}))
run_dir = Path(solver_b.last_checkpoint.path).parent
journal = ResourceJournal(run_dir)
journal.begin(f"{solver_b.run_id}:iteration:3", attempt_id="crash-demo", upper_bound=160, kind="optimizer_iteration")
recovered = restore(solver_b.last_checkpoint.path)
recovered.resume()
print("recovered:", recovered.status, recovered.resources)
assert recovered.resources["known_spent"] == 544
assert recovered.resources["uncertain_upper_charge"] == 160
assert recovered.resources["total_charged"] == 704


recovered: stopped {'shots_used': 704, 'known_spent': 544, 'uncertain_upper_charge': 160, 'total_charged': 704, 'training_shots': 480, 'logical_training_shots': 480, 'readout_shots': 64, 'readout_reserved': 0, 'shots_available': 0, 'objective_evaluations': 15, 'optimizer_steps': 3, 'num_stages': 4, 'num_treads': 1, 'checkpoint_writes': 5}


## D. Capability matrix

An unsupported component is rejected explicitly; it does not turn into a fake
partial checkpoint.

In [5]:
table = capabilities()
for name in ("qaoa_cpu", "qaoa_gpu", "rqaoa_cpu", "ls_rqaoa_cpu", "qaoa_in_qaoa_cpu", "exact_cpu", "gw_cpu", "simulated_annealing_cpu"):
    item = table[name]
    print(f"{name:24s} supported={item['supports_checkpoint']!s:5s} boundaries={item['boundaries']}")


qaoa_cpu                 supported=True  boundaries=['completed ADAM iteration', 'training_done', 'readout_done']
qaoa_gpu                 supported=False boundaries=[]
rqaoa_cpu                supported=False boundaries=[]
ls_rqaoa_cpu             supported=False boundaries=[]
qaoa_in_qaoa_cpu         supported=False boundaries=[]
exact_cpu                supported=False boundaries=[]
gw_cpu                   supported=False boundaries=[]
simulated_annealing_cpu  supported=False boundaries=[]


## E. Generation size and budgets

A checkpoint contains a manifest, state.json and four numeric `.npy` sidecars for
`theta/moment/variance/best_theta`. A large statevector is not saved by default.

In [6]:
run_dir = Path(continued.last_checkpoint.path).parent
gen_sizes = []
for gen in sorted(run_dir.glob("gen_*")):
    size = sum(p.stat().st_size for p in gen.iterdir() if p.is_file())
    gen_sizes.append((gen.name, size))
print("generations:", gen_sizes)
print("total bytes:", sum(v for _, v in gen_sizes))


generations: [('gen_000001', 8612), ('gen_000002', 10282), ('gen_000003', 12044), ('gen_000004', 13683), ('gen_000005', 13686), ('gen_000006', 13268)]
total bytes: 71575


## F. Lineage and spending plot

The plot is built from the saved manifests; the solver is not called.

In [7]:
import json
manifest_rows = []
for gen in sorted(run_dir.glob("gen_*")):
    data = json.loads((gen / "manifest.json").read_text(encoding="utf-8"))
    manifest_rows.append((data["generation"], data.get("phase"), data.get("journal_cutoff", {}).get("known_spent", 0)))
fig, ax = plt.subplots(figsize=(9, 4))
xs = [row[0] for row in manifest_rows]
ys = [row[2] for row in manifest_rows]
ax.step(xs, ys, where="post", marker="o", color="#2563eb")
ax.set_xlabel("generation")
ax.set_ylabel("known training shots")
ax.set_title("Checkpoint lineage and accepted resource prefix")
ax.grid(alpha=0.25)
fig.tight_layout()
figure_path = Path("docs/notebook_previews/11_checkpoints_and_resume.png")
figure_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_path, dpi=140, facecolor="white")
plt.close(fig)
print("saved:", figure_path)


saved: docs\notebook_previews\11_checkpoints_and_resume.png


## Summary

- `solve()` is always a new run.
- pause → inspect → restore → resume continues the same run.
- After two iterations: 320 training shots, 64 held, 320 available.
- A completed readout does not repeat the M measurements.
- After a crash the uncertain charge is neither zero nor doubled at reconciliation.
- GPU/RQAOA/LS/hierarchy checkpoints are explicitly not claimed in a10 and require
  the corresponding codecs.